# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dict
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list all record sets by their `@id`, and show the fields (columns) they contain with their own `@id` values.

In [ ]:
# List available record sets and their fields by `@id`

record_sets = list(dataset.record_sets())  # List of mlc.RecordSet objects
if not record_sets:
    print("No record sets found in this dataset. Please check the Croissant schema for 'recordSet' definitions.")
else:
    for rset in record_sets:
        print(f"Record Set: {rset.id}")  # rset.id gives @id
        print("  Fields:")
        for field in rset.fields:
            print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
        print('')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s found above.

All references to entities use their unique `@id`.

In [ ]:
# Extract data from ALL available record sets (by @id)
rs_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in rs_ids:
    print(f"Loading records for Record Set @id: {rs_id}")
    recs = list(dataset.records(record_set=rs_id))
    if len(recs) == 0:
        print(f"  No records found for {rs_id}.")
    else:
        df = pd.DataFrame(recs)
        dataframes[rs_id] = df
        print(f"  Columns: {df.columns.tolist()}")

# For demonstration, print columns and head for the first available record set with data
first_with_data = None
for rs_id in rs_ids:
    if rs_id in dataframes:
        first_with_data = rs_id
        break

if first_with_data:
    print(f"\nFirst record set with data: {first_with_data}")
    print("Columns:", dataframes[first_with_data].columns.tolist())
    display(dataframes[first_with_data].head())
else:
    print("No dataframes loaded. Please check that the Croissant schema defines record sets with accessible data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

- Remove outliers
- Normalize numeric fields
- Group data by key attributes

All column accesses below use their `@id` (not just label).

In [ ]:
# For EDA, use the first dataframe with data loaded previously
import numpy as np

if not dataframes:
    print("No data loaded. Cannot perform EDA.")
else:
    df = dataframes[first_with_data]
    numeric_field_id = None

    # Automatically find a numeric column by testing dtypes
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found in the record set for EDA.")
    else:
        print(f"Numeric field selected for filtering: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.90) if len(df) > 0 else 0
        # Filter out outliers (greater than 90th percentile)
        filtered_df = df[df[numeric_field_id] <= threshold]
        print(f"Filtered records with {numeric_field_id} <= {threshold:.2f} (removing top 10% values):")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - mean) / std if std > 0 else filtered_df[numeric_field_id] - mean
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Attempt grouping by a categorical field if available
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < 20:
                group_field_id = col
                break
        if group_field_id:
            grouped = filtered_df.groupby(group_field_id, dropna=True)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field_id} (showing group means):")
            display(grouped.head())
        else:
            print("No suitable categorical/grouping field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Below, a histogram and (if possible) boxplot/grouped bar is produced for selected fields.

All columns are referenced by their `@id`.

In [ ]:
# Simple visualization for the selected numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes or numeric_field_id is None:
    print("Insufficient data for visualization.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If group_field_id found above, do a boxplot or barplot
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we demonstrated how to use the `mlcroissant` library to discover and load data from a Croissant-defined dataset using only entity `@id`s for all references. We linked record sets and fields by `@id`, dynamically selected fields for analysis, and performed basic filtering, normalization, grouping, and visualization.

Key points:
- The dataset provides ordered logistic regression results from household surveys in Northern Kenya, tracking factors affecting knowledge and rangeland management practices.
- All entities (record sets, fields/columns) were referenced by unique `@id` as per FAIR best practices.
- Data was filtered for outliers, normalized, and grouped for simple statistical EDA and visualized with histograms/boxplots.

**Further analysis can be performed by examining additional record sets, fields, and relationships as defined in the Croissant schema.**